# Stage 2 models and targeted tuning
This notebook preserves the Stage 1 fit and validation indices. All tuning cross-validation runs occur inside the fit partition with fingerprint groups kept together. The official test dataset is not loaded.

In [ ]:
from pathlib import Path
import json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

ROOT = Path('..').resolve()
DATA, RESULTS, MODELS = ROOT / 'data' / 'processed', ROOT / 'results', ROOT / 'models'
PLOTS = RESULTS / 'plots'
SEED, THREADS = 42, 2
CLASSES = ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers']
for filename in ['stage2_model_comparison.csv', 'stage2_experiment_log.csv', 'stage2_tuning_log.csv']:
    assert not (RESULTS / filename).exists(), f'Existing {filename} requires review before rerun.'
frame = pd.read_csv(DATA / 'selected_training.csv')
fit_idx = pd.read_csv(DATA / 'fit_indices.csv')['row_index'].to_numpy()
val_idx = pd.read_csv(DATA / 'validation_indices.csv')['row_index'].to_numpy()
assert len(set(fit_idx) & set(val_idx)) == 0 and set(fit_idx) | set(val_idx) == set(range(len(frame)))
predictors = [c for c in frame if c not in {'id', 'label', 'attack_cat', '_fingerprint'}]
assert {'id', 'label', 'attack_cat', '_fingerprint'}.isdisjoint(predictors)
X_fit, X_val = frame.iloc[fit_idx][predictors], frame.iloc[val_idx][predictors]
y_fit, y_val = frame.iloc[fit_idx]['attack_cat'], frame.iloc[val_idx]['attack_cat']
numeric = X_fit.select_dtypes(include='number').columns.tolist()
categorical = X_fit.select_dtypes(exclude='number').columns.tolist()
fingerprints = pd.util.hash_pandas_object(X_fit, index=False, categorize=True).astype('uint64')
cv = list(StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED).split(X_fit, y_fit, fingerprints))
assert all(set(fingerprints.iloc[tr]).isdisjoint(set(fingerprints.iloc[te])) for tr, te in cv)
PLOTS.mkdir(parents=True, exist_ok=True); MODELS.mkdir(parents=True, exist_ok=True)
print(X_fit.shape, X_val.shape, len(numeric), len(categorical), [len(te) for _, te in cv])

In [ ]:
def preprocessor(dense=False, scale_numeric=False):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(num_steps), numeric),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=not dense, dtype=np.float32))]), categorical),
    ], sparse_threshold=0 if dense else 1)

validation_rows, class_rows, tuning_rows, fitted = [], [], [], {}
def persist():
    pd.DataFrame(validation_rows).to_csv(RESULTS / 'stage2_model_comparison.csv', index=False)
    pd.DataFrame(class_rows).to_csv(RESULTS / 'stage2_experiment_log.csv', index=False)
    pd.DataFrame(tuning_rows).to_csv(RESULTS / 'stage2_tuning_log.csv', index=False)

def plot_matrix(name, matrix):
    fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(matrix, cmap='Blues')
    ax.set(xticks=range(6), yticks=range(6), xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted', ylabel='Actual', title=name)
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    for i in range(6):
        for j in range(6): ax.text(j, i, str(matrix[i,j]), ha='center', va='center', fontsize=8)
    fig.tight_layout(); fig.savefig(PLOTS / f'stage2_{name.lower().replace(' ', '_')}_confusion.png', dpi=130); plt.close(fig)

def evaluate(name, pipeline, config, phase='baseline', sample_weight=None):
    start = time.perf_counter()
    fit_args = {'classifier__sample_weight': sample_weight} if sample_weight is not None else {}
    pipeline.fit(X_fit, y_fit, **fit_args)
    seconds = time.perf_counter() - start
    pred = pipeline.predict(X_val)
    report = classification_report(y_val, pred, labels=CLASSES, output_dict=True, zero_division=0)
    matrix = confusion_matrix(y_val, pred, labels=CLASSES)
    attack_true, attack_pred = y_val.to_numpy() != 'Normal', pred != 'Normal'
    row = {'name': name, 'phase': phase, 'configuration': json.dumps(config, sort_keys=True, default=str), 'fit_seconds': seconds, 'accuracy': accuracy_score(y_val, pred), 'macro_f1': f1_score(y_val, pred, average='macro'), 'weighted_f1': f1_score(y_val, pred, average='weighted'), 'attack_recall': float(attack_pred[attack_true].mean()), 'normal_false_positive_rate': float(attack_pred[~attack_true].mean()), 'confusion_matrix': json.dumps(matrix.tolist())}
    for label in CLASSES:
        for metric in ['precision', 'recall', 'f1-score']: row[f'{label}_{metric}'] = report[label][metric]
        class_rows.append({'name':name, 'phase':phase, 'configuration':row['configuration'], 'class':label, 'support':int(report[label]['support']), 'precision':report[label]['precision'], 'recall':report[label]['recall'], 'f1':report[label]['f1-score'], 'fit_seconds':seconds})
    validation_rows.append(row); fitted[name] = pipeline; plot_matrix(name, matrix); persist()
    print(name, round(row['macro_f1'],4), round(row['Analysis_recall'],3), round(row['Backdoor_recall'],3), round(row['DoS_recall'],3), round(row['normal_false_positive_rate'],3), round(seconds,1), flush=True)
    return row

def cv_score(name, builder, config, weighted=False):
    scores, started = [], time.perf_counter()
    for tr, te in cv:
        pipe = builder()
        weights = compute_sample_weight('balanced', y_fit.iloc[tr]) if weighted else None
        args = {'classifier__sample_weight': weights} if weights is not None else {}
        pipe.fit(X_fit.iloc[tr], y_fit.iloc[tr], **args)
        scores.append(f1_score(y_fit.iloc[te], pipe.predict(X_fit.iloc[te]), average='macro'))
    row = {'name':name, 'configuration':json.dumps(config, sort_keys=True, default=str), 'cv_folds':2, 'cv_macro_f1_mean':float(np.mean(scores)), 'cv_macro_f1_std':float(np.std(scores)), 'fit_seconds':time.perf_counter()-started, 'weighted':weighted}
    tuning_rows.append(row); persist(); print('CV', name, round(row['cv_macro_f1_mean'],4), round(row['fit_seconds'],1), flush=True); return row

## Additional baselines
Logistic Regression scales numeric values. HistGradientBoosting uses a dense design matrix. XGBoost uses the same train-only transformation and two CPU threads.

In [ ]:
evaluate('Logistic Regression', Pipeline([('preprocessor', preprocessor(scale_numeric=True)), ('classifier', LogisticRegression(max_iter=250, C=1.0, solver='saga', n_jobs=THREADS, random_state=SEED, tol=2e-3))]), {'C':1.0, 'class_weight':None, 'max_iter':250, 'solver':'saga'})
evaluate('Logistic Regression balanced', Pipeline([('preprocessor', preprocessor(scale_numeric=True)), ('classifier', LogisticRegression(max_iter=250, C=1.0, solver='saga', n_jobs=THREADS, random_state=SEED, tol=2e-3, class_weight='balanced'))]), {'C':1.0, 'class_weight':'balanced', 'max_iter':250, 'solver':'saga'})
evaluate('Bagging Tree', Pipeline([('preprocessor', preprocessor()), ('classifier', BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=18, min_samples_leaf=2, random_state=SEED), n_estimators=40, max_samples=0.7, n_jobs=THREADS, random_state=SEED))]), {'n_estimators':40, 'max_samples':0.7, 'base_max_depth':18, 'base_min_samples_leaf':2})
evaluate('AdaBoost', Pipeline([('preprocessor', preprocessor()), ('classifier', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, random_state=SEED), n_estimators=100, learning_rate=0.5, random_state=SEED))]), {'n_estimators':100, 'learning_rate':0.5, 'base_max_depth':2})
encoder = LabelEncoder().fit(CLASSES)
y_fit_xgb, y_val_xgb = encoder.transform(y_fit), encoder.transform(y_val)
xgb_pipe = Pipeline([('preprocessor', preprocessor()), ('classifier', XGBClassifier(objective='multi:softprob', num_class=6, n_estimators=120, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=THREADS, random_state=SEED, eval_metric='mlogloss'))])
start=time.perf_counter(); xgb_pipe.fit(X_fit,y_fit_xgb); seconds=time.perf_counter()-start; pred=encoder.inverse_transform(xgb_pipe.predict(X_val).astype(int))
report=classification_report(y_val,pred,labels=CLASSES,output_dict=True,zero_division=0); matrix=confusion_matrix(y_val,pred,labels=CLASSES); attack_true=y_val.to_numpy()!='Normal'; attack_pred=pred!='Normal'
row={'name':'XGBoost','phase':'baseline','configuration':json.dumps({'n_estimators':120,'max_depth':6,'learning_rate':0.1,'subsample':0.8,'colsample_bytree':0.8,'tree_method':'hist','n_jobs':2}), 'fit_seconds':seconds,'accuracy':accuracy_score(y_val,pred),'macro_f1':f1_score(y_val,pred,average='macro'),'weighted_f1':f1_score(y_val,pred,average='weighted'),'attack_recall':float(attack_pred[attack_true].mean()),'normal_false_positive_rate':float(attack_pred[~attack_true].mean()),'confusion_matrix':json.dumps(matrix.tolist())}
for label in CLASSES:
    for metric in ['precision','recall','f1-score']: row[f'{label}_{metric}']=report[label][metric]
    class_rows.append({'name':'XGBoost','phase':'baseline','configuration':row['configuration'],'class':label,'support':int(report[label]['support']),'precision':report[label]['precision'],'recall':report[label]['recall'],'f1':report[label]['f1-score'],'fit_seconds':seconds})
validation_rows.append(row); fitted['XGBoost']=xgb_pipe; plot_matrix('XGBoost',matrix); persist(); print('XGBoost',round(row['macro_f1'],4),round(row['Analysis_recall'],3),round(row['Backdoor_recall'],3),round(row['DoS_recall'],3),round(row['normal_false_positive_rate'],3),round(seconds,1),flush=True)

## Targeted grouped cross-validation
Each candidate uses only fit rows. Validation metrics appear only after selecting the highest mean cross-validation macro-F1 configuration for each model family.

In [ ]:
dt_candidates=[{'max_depth':None,'min_samples_leaf':1,'class_weight':None},{'max_depth':18,'min_samples_leaf':2,'class_weight':'balanced'},{'max_depth':24,'min_samples_leaf':3,'class_weight':'balanced'}]
dt_cv=[]
for cfg in dt_candidates:
    build=lambda cfg=cfg: Pipeline([('preprocessor',preprocessor()),('classifier',DecisionTreeClassifier(random_state=SEED,**cfg))])
    dt_cv.append((cv_score('Decision Tree tuning',build,cfg,weighted=False),cfg,build))
best_dt=max(dt_cv,key=lambda x:x[0]['cv_macro_f1_mean']); evaluate('Decision Tree tuned',best_dt[2](),best_dt[1],phase='tuned')

hgb_candidates=[({'max_iter':150,'learning_rate':0.08,'max_leaf_nodes':31,'min_samples_leaf':20,'l2_regularization':0.1},False),({'max_iter':150,'learning_rate':0.08,'max_leaf_nodes':31,'min_samples_leaf':20,'l2_regularization':0.1},True),({'max_iter':200,'learning_rate':0.06,'max_leaf_nodes':15,'min_samples_leaf':30,'l2_regularization':0.5},True)]
hgb_cv=[]
for cfg,weighted in hgb_candidates:
    build=lambda cfg=cfg: Pipeline([('preprocessor',preprocessor(dense=True)),('classifier',HistGradientBoostingClassifier(random_state=SEED,**cfg))])
    hgb_cv.append((cv_score('HistGradientBoosting tuning',build,{**cfg,'sample_weight':'balanced' if weighted else None},weighted),cfg,weighted,build))
best_hgb=max(hgb_cv,key=lambda x:x[0]['cv_macro_f1_mean']); weights=compute_sample_weight('balanced',y_fit) if best_hgb[2] else None; evaluate('HistGradientBoosting tuned',best_hgb[3](),{**best_hgb[1],'sample_weight':'balanced' if best_hgb[2] else None},phase='tuned',sample_weight=weights)

rf_candidates=[{'n_estimators':50,'max_depth':None,'min_samples_leaf':1,'class_weight':None},{'n_estimators':50,'max_depth':None,'min_samples_leaf':1,'class_weight':'balanced_subsample'},{'n_estimators':50,'max_depth':22,'min_samples_leaf':2,'class_weight':'balanced_subsample'}]
rf_cv=[]
for cfg in rf_candidates:
    build=lambda cfg=cfg: Pipeline([('preprocessor',preprocessor()),('classifier',RandomForestClassifier(random_state=SEED,n_jobs=THREADS,**cfg))])
    rf_cv.append((cv_score('Random Forest tuning',build,cfg,weighted=False),cfg,build))
best_rf=max(rf_cv,key=lambda x:x[0]['cv_macro_f1_mean']); evaluate('Random Forest tuned',best_rf[2](),best_rf[1],phase='tuned')

In [ ]:
results=pd.DataFrame(validation_rows).sort_values('macro_f1',ascending=False).reset_index(drop=True)
best=results.iloc[0]; results['recommendation']=np.where(results.name==best['name'],'KEEP','TUNE')
results.loc[results['macro_f1']<0.2,'recommendation']='DISCARD'
results['reason']=results.apply(lambda r: 'Highest Stage 2 validation macro-F1' if r['name']==best['name'] else ('Poor macro-F1 or attack coverage' if r['recommendation']=='DISCARD' else 'Candidate for parameter or threshold tuning'),axis=1)
results.to_csv(RESULTS/'stage2_model_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(11,4)); results.set_index('name')[['macro_f1','Analysis_recall','Backdoor_recall','DoS_recall','normal_false_positive_rate']].plot.bar(ax=ax,rot=25); ax.set_ylim(0,1); ax.set_ylabel('Score / rate'); fig.tight_layout(); fig.savefig(PLOTS/'stage2_ranked_minority_performance.png',dpi=140); plt.close(fig)
best_pipeline=fitted[best['name']]
joblib.dump(best_pipeline,MODELS/'stage2_best_validation_pipeline.joblib')
(MODELS/'stage2_best_validation_metadata.json').write_text(json.dumps({'selected_model':best['name'],'selection_metric':'validation macro-F1','validation_macro_f1':float(best['macro_f1']),'classes':CLASSES,'feature_list':predictors,'fit_rows':len(X_fit),'validation_rows':len(X_val),'validation_indices_source':'data/processed/validation_indices.csv'},indent=2),encoding='utf-8')
assert (MODELS/'stage2_best_validation_pipeline.joblib').is_file() and (MODELS/'stage2_best_validation_metadata.json').is_file()
display(results[['name','phase','macro_f1','accuracy','Analysis_recall','Backdoor_recall','DoS_recall','normal_false_positive_rate','fit_seconds','recommendation','reason']])